# Planning

**Prerequisites:** `03_retries_and_reflection.ipynb`, and from `01_foundations`: `05_structured_outputs.ipynb`, `08_asyncio.ipynb`

ReAct decides one step at a time. Every turn, the whole conversation goes back to the model and it works out what to do next from scratch. For a two-step task that's fine. For a six-step task it's six full-context calls to answer a question whose shape was obvious from the start.

Deciding the whole sequence up front changes that, and it changes something more interesting too. A list of steps written in advance is a *structure* — you can look at it before running any of it, and see that step 1 needs step 0's result while step 2 needs nothing from either. ReAct can never see that, because it only ever knows about the step it's on.

This notebook builds the plan, then executes it twice: once step by step, and once as the graph it always was.

**By the end you'll have:**
- Built `PlannerThink`, producing a whole sequence of tool calls in one structured call
- Solved the problem that makes planning hard: writing an argument whose value won't exist until an earlier step runs, via `$stepN` and `$stepN.field` references resolved mechanically at execution time
- Seen why counting *decided* steps and counting *executed* steps have to be two different functions
- Run the plan sequentially with `Loop(until=plan_done)`, and watched `DebugThink` quietly repair two arguments the planner got wrong
- Extracted the dependency graph from the plan's own `$stepN` references, and run the same plan through `Graph` — each step starting the moment its own dependencies finish, with a real measured speedup

## Setup

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
sys.path.append(str(Path.cwd().parent.parent / "01_foundations"))

import copy
import json
import time

from chat import Conversation
from runtime import Runtime, Loop, Graph
from observe import Observe
from think import PlannerThink, plan_done, plan_progress, step_dependencies, resolve_args
from act import Act

Three tools this time. `get_weather` is new and matters for two reasons: it returns a **JSON object with named fields** rather than a single value, and — along with `get_current_time` — it sleeps for a moment, so that running things at the same time is something you can measure rather than take on faith.

In [2]:
from datetime import datetime
from typing import Literal
from zoneinfo import ZoneInfo, ZoneInfoNotFoundError

from pydantic import BaseModel, field_validator

WEATHER = {"Tokyo": {"temp_f": 88.0, "condition": "clear"},
           "London": {"temp_f": 61.0, "condition": "overcast"}}

def get_weather(city):
    time.sleep(1.5)                      # stand-in for a real network call
    return json.dumps(WEATHER[city])

def get_current_time(timezone):
    time.sleep(1.5)
    return datetime.now(ZoneInfo(timezone)).strftime("%Y-%m-%d %H:%M %Z")

def convert_temperature(value, from_unit, to_unit):
    if from_unit.upper() == "F" and to_unit.upper() == "C":
        return round((value - 32) * 5 / 9, 1)
    if from_unit.upper() == "C" and to_unit.upper() == "F":
        return round(value * 9 / 5 + 32, 1)
    raise ValueError(f"unsupported conversion: {from_unit} to {to_unit}")

class GetWeatherArgs(BaseModel):
    city: Literal["Tokyo", "London"]

class GetCurrentTimeArgs(BaseModel):
    timezone: str

    @field_validator("timezone")
    @classmethod
    def timezone_must_be_valid(cls, v):
        try:
            ZoneInfo(v)
        except ZoneInfoNotFoundError:
            raise ValueError(f"'{v}' is not a valid IANA timezone")
        return v

class ConvertTemperatureArgs(BaseModel):
    value: float
    from_unit: Literal["C", "F"]
    to_unit: Literal["C", "F"]

FUNCTIONS = {"get_weather": get_weather, "get_current_time": get_current_time,
             "convert_temperature": convert_temperature}
SCHEMAS = {"get_weather": GetWeatherArgs, "get_current_time": GetCurrentTimeArgs,
           "convert_temperature": ConvertTemperatureArgs}

tools = [
    {"type": "function", "function": {
        "name": "get_weather",
        "description": "Get the current weather in a city.",
        "parameters": {"type": "object",
                       "properties": {"city": {"type": "string", "enum": ["Tokyo", "London"]}},
                       "required": ["city"]}}},
    {"type": "function", "function": {
        "name": "get_current_time",
        "description": "Get the current date and time in a given IANA timezone.",
        "parameters": {"type": "object",
                       "properties": {"timezone": {"type": "string", "description": "IANA timezone name"}},
                       "required": ["timezone"]}}},
    {"type": "function", "function": {
        "name": "convert_temperature",
        "description": "Convert a temperature value between Celsius and Fahrenheit.",
        "parameters": {"type": "object",
                       "properties": {"value": {"type": "number"},
                                      "from_unit": {"type": "string", "enum": ["C", "F"]},
                                      "to_unit": {"type": "string", "enum": ["C", "F"]}},
                       "required": ["value", "from_unit", "to_unit"]}}},
]

# what each tool returns -- the planner needs this to reference a field of a result it hasn't seen
OUTPUTS = {
    "get_weather": 'a JSON object with fields "temp_f" (number, degrees Fahrenheit) and "condition" (string)',
    "get_current_time": "a single formatted date-time string",
    "convert_temperature": "a single number, the converted temperature",
}

REQUEST = "What's the current time in Tokyo, and what's the temperature there in Celsius?"

## The argument that doesn't exist yet

Planning runs into one problem immediately, and everything else about `PlannerThink` is shaped by it.

To answer the request, something has to convert Tokyo's temperature to Celsius. But the planner is writing the plan *before* anything runs — it has no idea what `get_weather` will return. It cannot write `{"value": 88.0, ...}`, because 88.0 is not knowable yet. And it must not guess, because a plausible guess is worse than an obvious placeholder.

So the plan gets a way to say "whatever step 1 returns":

- `"$step0"` — the entire result of step 0
- `"$step1.temp_f"` — just the `temp_f` field, when that step returns a JSON object

This is why `PlannerThink` takes an `outputs` dict. To reference `.temp_f` the planner has to know `get_weather` returns something with a `temp_f` in it, and no tool *schema* describes a return value — schemas describe arguments. `_tool_descriptions()` stitches the two together:

```python
def _tool_descriptions(self):
    return "\n".join(
        f"- {t['function']['name']}({', '.join(t['function']['parameters']['properties'])}) "
        f"returns: {self.outputs.get(t['function']['name'], 'a single value')}"
        for t in self.tools
    )
```

The plan itself comes back through `extract()` (chapter 1, notebook 5) against a Pydantic model, so it's validated JSON before anything looks at it:

```python
class PlanStep(BaseModel):
    tool: str
    args: dict[str, str]

class Plan(BaseModel):
    steps: list[PlanStep]
```

Note `args: dict[str, str]` — every argument is a string, including numbers. That's deliberate: `"$step1.temp_f"` and `"100"` have to live in the same field, so the type has to accommodate the reference. The real values get restored when the reference is resolved, and anything still wrong by then is `DebugThink`'s problem.

Let's make a plan and look at it:

In [3]:
planner = PlannerThink(tools=tools, outputs=OUTPUTS)

planned = Conversation()
planned.messages.append({"role": "user", "content": REQUEST})
first_decision = planner.run(planned)      # appends the plan, returns step 0's decision
planned.messages.append(first_decision)

plan = next(m for m in planned.messages if m["role"] == "plan")
for i, step in enumerate(plan["steps"]):
    print(f"step{i}: {step['tool']}({step['args']})")

step0: get_current_time({'timezone': 'Tokyo'})
step1: get_weather({'city': 'Tokyo'})
step2: convert_temperature({'value': '$step1.temp_f', 'from_unit': 'Fahrenheit', 'to_unit': 'Celsius'})


Three steps, and one of them carries `"$step1.temp_f"` — the planner correctly worked out that it needs a number it can't have yet, and that the number is one field of step 1's object rather than the whole thing. Forwarding the whole JSON object would have failed validation: `ConvertTemperatureArgs.value` is a `float`.

Resolution happens later, mechanically, in `Observe`:

```python
def _resolve(self, value, conversation):
    match = re.fullmatch(r"\$(?:step)?(\d+)(?:\.(\w+))?", value) if isinstance(value, str) else None
    if match is None:
        return value
    index, field = match.groups()
    raw = conversation.step_outputs[int(index)]
    return json.loads(raw)[field] if field is not None else raw
```

A regex and a dictionary lookup. No model call, no judgment — the decision about what to reference was made once, by the planner, at planning time. This is the same division as everywhere else in the chapter: `Think` decides, everything downstream executes what was decided.

And this is what `step_outputs` was for. Notebook 2 introduced it as an indexed list of every raw tool result, deliberately kept out of `messages`. `$step1` indexes straight into it — which is exactly why raw results had to be kept somewhere. The *observation* in `messages` says "The weather is clear, 88.0 degrees Fahrenheit"; you cannot reliably get a float back out of a sentence. `step_outputs[1]` still holds `'{"temp_f": 88.0, "condition": "clear"}'`.

## Decided is not executed

Two different questions get asked about a plan in progress, and conflating them causes a real bug.

```python
def plan_progress(conversation):
    plan_index, plan = _latest_plan(conversation)
    if plan is None:
        return None, 0
    decided = sum(1 for m in conversation.messages[plan_index + 1:] if m.get("role") == "decision")
    return plan, decided

def plan_done(conversation):
    plan_index, plan = _latest_plan(conversation)
    if plan is None:
        return True
    executed = sum(1 for m in conversation.messages[plan_index + 1:] if m.get("role") == "tool")
    return executed >= len(plan["steps"])
```

`plan_progress` counts **decisions** — it answers "which step should I set up next?", and `Observe` uses it to emit the following step's tool call the instant it finishes reporting the current one.

`plan_done` counts **executed** steps — `tool` observations that actually exist.

The two differ by exactly one at the most important moment: right after the last step's decision has been produced but before `Act` has run it. A version of `plan_done` that counted decisions would return `True` there, ending the loop with the final step decided and never executed.

Both scan from `_latest_plan`, the *last* plan message rather than the first, which is what makes replanning work: add a second plan and the counters start over against it.

## Running the plan, one step at a time

The declared loop is smaller than ReAct's, because the plan does the work that `Think` was doing every turn:

```python
Runtime(loop=[Loop([Act(...), Observe(...)], until=plan_done), PlannerThink(...)], repeat_from=0)
```

`Act` executes the current decision; `Observe` reports it and emits the next one. Those two components are the entire execution engine — no `Think` in sight, because there is nothing left to decide until the plan runs out.

`until=plan_done` is where `Loop`'s predicate form earns itself. Notebook 3 used `count`, a fixed number of rounds. Here the number of rounds is however many steps the plan happens to have, which nothing knows in advance — so the stopping condition has to be a question asked of the conversation.

The conversation already carries the plan and step 0's decision from the cell above, so this picks up from there:

In [4]:
sequential = copy.deepcopy(planned)

start = time.perf_counter()
Runtime(
    loop=[
        Loop([Act(functions=FUNCTIONS, schemas=SCHEMAS), Observe()], until=plan_done),
        PlannerThink(tools=tools, outputs=OUTPUTS),
    ],
    repeat_from=0,
).run(sequential)
sequential_seconds = time.perf_counter() - start

for m in sequential.messages:
    print(f"- {m['role']:9} {m['content']!r}")
print(f"\nwall clock: {sequential_seconds:.1f}s")

- user      "What's the current time in Tokyo, and what's the temperature there in Celsius?"
- plan      '{"steps":[{"tool":"get_current_time","args":{"timezone":"Tokyo"}},{"tool":"get_weather","args":{"city":"Tokyo"}},{"tool":"convert_temperature","args":{"value":"$step1.temp_f","from_unit":"Fahrenheit","to_unit":"Celsius"}}]}'
- decision  '{"timezone": "Tokyo"}'
- tool      'The current time in Tokyo is 20:11.'
- decision  '{"city": "Tokyo"}'
- tool      'The temperature in Tokyo is 88.0 degrees Fahrenheit.'
- decision  '{"value": 88.0, "from_unit": "Fahrenheit", "to_unit": "Celsius"}'
- tool      'The temperature in Tokyo is 31.1 degrees Celsius.'
- decision  'The current time in Tokyo is 20:11 and the temperature is 31.1 degrees Celsius.'
- assistant 'The current time in Tokyo is 20:11 and the temperature is 31.1 degrees Celsius.'

wall clock: 5.3s


The plan was followed exactly, `$step1.temp_f` became a real number, and the run finished with an answer.

Two things in that transcript are worth stopping on.

**The planner got two arguments wrong, and you can't tell from the output.** Look at the plan again: `{"timezone": "Tokyo"}` — not a valid IANA timezone, `GetCurrentTimeArgs`'s validator rejects it. And `{"from_unit": "Fahrenheit"}` — not `"C"` or `"F"`, so `ConvertTemperatureArgs` rejects that too. Both calls failed on their first attempt, and `DebugThink` (notebook 3) repaired both inside `Act.execute()` without a word reaching the conversation. That's the design working: two failures, two recoveries, zero context spent, and nothing for `_doom_loop` to misread.

**`PlannerThink` ran a second time at the end, and did something different.** It has two jobs, and picks between them by asking whether a plan already exists:

```python
def run(self, conversation):
    plan, _ = plan_progress(conversation)
    if plan is None:
        return self._make_plan(conversation)
    return self._continue_or_finish(conversation)
```

Once every planned step has run, the second job asks whether the plan was actually enough:

```python
class ContinueOrFinish(BaseModel):
    done: bool
    final_answer: str = ""
    additional_steps: list[PlanStep] = []
```

`done: true` and it writes the final answer. `done: false` and it appends more steps — a second plan message, which `_latest_plan` now points at, so `plan_done` resets and `repeat_from=0` sends control back into the same `Loop` to run them.

This matters because planning up front is a bet. ReAct re-decides every turn and can't be blindsided; a planner commits to a guess about a task it hasn't started. Replanning is what makes that bet safe to take — and the new steps have to keep counting `$stepN` from where the old ones left off, which is why the prompt tells the planner exactly how many results already exist and what index to continue from.

## The plan was already a graph

Now the part that ReAct structurally cannot do.

Look at the plan once more, but at the dependencies instead of the order. `step_dependencies` reads them straight out of the arguments — a step depends on step N exactly when one of its arguments references `$stepN`:

```python
def step_dependencies(step):
    deps = set()
    for value in step["args"].values():
        match = _STEP_REF.fullmatch(value) if isinstance(value, str) else None
        if match is not None:
            deps.add(int(match.group(1)))
    return deps
```

In [5]:
for i, step in enumerate(plan["steps"]):
    deps = step_dependencies(step)
    print(f"step{i}: {step['tool']:20} depends on {sorted(deps) if deps else 'nothing'}")

step0: get_current_time     depends on nothing
step1: get_weather          depends on nothing
step2: convert_temperature  depends on [1]


The plan is written as a list because lists are how you write things down, but it never was one. Two of those steps depend on nothing at all — getting the time in Tokyo and getting the weather in Tokyo have no relationship to each other whatsoever. Running one before the other is an accident of how the text was ordered.

`$stepN` didn't just solve the unknown-argument problem. It **is** the dependency edge — the planner had to state, precisely, which earlier results each step needs, and having stated it, the graph is already there to be read.

`Graph` executes against that structure instead of the ordering:

```python
class Graph:
    def run(self, conversation):
        plan, _ = plan_progress(conversation)
        steps = plan["steps"]
        deps = [step_dependencies(step) for step in steps]
        results, started = {}, set()

        with concurrent.futures.ThreadPoolExecutor(max_workers=self.max_workers) as pool:
            in_flight = {}

            def launch_ready():
                for i, step in enumerate(steps):
                    if i in started or not (deps[i] <= results.keys()):
                        continue
                    started.add(i)
                    args = resolve_args(step, results)
                    in_flight[pool.submit(self.act.execute, step["tool"], json.dumps(args))] = i

            launch_ready()
            while in_flight:
                finished = next(concurrent.futures.as_completed(in_flight))
                i = in_flight.pop(finished)
                effect = finished.result()
                results[i] = effect["content"]
                conversation.messages.append({"role": "tool", "content": self._phrase(effect, conversation)})
                launch_ready()
        return None
```

`launch_ready()` starts every step whose dependencies are all in `results`. Then the loop waits for whichever step finishes *first* — `as_completed` — records it, and calls `launch_ready()` again. So a step begins the instant the specific things it was waiting for are done, regardless of what else is still running.

`resolve_args` is `Observe._resolve`'s counterpart, reading from the local `results` dict rather than `step_outputs`, because these steps didn't run in index order and `step_outputs` is append-ordered.

Same plan, executed both ways:

In [6]:
act = Act(functions=FUNCTIONS, schemas=SCHEMAS)

# one at a time, in the order written
start = time.perf_counter()
results = {}
for i, step in enumerate(plan["steps"]):
    results[i] = act.execute(step["tool"], json.dumps(resolve_args(step, results)))["content"]
one_at_a_time = time.perf_counter() - start

# the same steps, scheduled by dependency
graphed = Conversation()
graphed.messages.append({"role": "user", "content": REQUEST})
graphed.messages.append(plan)

start = time.perf_counter()
Graph(act=act).run(graphed)
graph_seconds = time.perf_counter() - start

print(f"one at a time: {one_at_a_time:.1f}s")
print(f"Graph:         {graph_seconds:.1f}s")
print()
for m in graphed.messages:
    if m["role"] == "tool":
        print(f"- tool {m['content']!r}")

one at a time: 3.6s
Graph:         2.5s

- tool 'The temperature in Tokyo is 88.0 degrees Fahrenheit.'
- tool 'The current time in Tokyo is 20:11.'
- tool 'The temperature in Tokyo is 31.1 degrees Celsius.'


The two independent steps overlapped, so the total is roughly the longest path through the graph rather than the sum of every step. And the observations come back in *completion* order, not plan order — direct evidence that they raced rather than queued.

### Threads here, not `asyncio`

Chapter 1's notebook 8 made the case for `asyncio` and proved a real speedup with `gather`. This chapter uses `concurrent.futures.ThreadPoolExecutor` instead, and the reason is practical rather than principled: notebooks already run inside an event loop, and library code that starts its own conflicts with it in ways that are genuinely unpleasant to debug. Threads sidestep that entirely.

Nothing is lost. Every one of these calls is blocking I/O — an HTTP request or a `sleep` — so the thread spends its time waiting, and waiting is exactly what releases the GIL. This is I/O-bound work, which threads handle just as well as coroutines.

### Why `Act.execute()` had to be pure

Notebook 3 mentioned in passing that `execute()` reads and writes nothing shared. Here is where that stops being a stylistic note. `pool.submit(self.act.execute, ...)` runs the *same* `Act` object from several threads at once. Had `execute()` accumulated anything on `self` — a retry counter, a log, the last error — those threads would be writing over each other, and the resulting bug would appear only under concurrency and only sometimes.

`execute()` creates its own `DebugThink` and its own `attempts` list per call, as locals. Every concurrent call gets its own. That was a deliberate choice made two notebooks before anything ran in parallel, and it's why nothing had to change here.

### What `Graph` costs

`Graph` doesn't replace the sequential path; it's a different trade.

It commits to the whole plan up front. `Observe`'s step-by-step execution emits one decision at a time, so a plan can be abandoned or replanned partway through. `Graph` launches everything whose dependencies are met, immediately — which is the entire point, and also means a step whose usefulness depended on an earlier result being surprising will run anyway.

It's also the one component in this chapter that imports from `think.py`. `Runtime` and `Loop` know nothing about what they're running; `Graph` needs `plan_progress`, `step_dependencies`, and `resolve_args`, because it schedules *plans* specifically. That's not a leak to fix — it's an honest admission that `Graph` is a plan executor, not a general primitive.

## What you built

✓ Built `PlannerThink`, producing an entire validated sequence of tool calls in one `extract()` call instead of one decision per turn

✓ Solved the unknown-argument problem with `$stepN` and `$stepN.field` references, and an `outputs` dict describing what each tool returns — since tool schemas describe arguments and never results

✓ Resolved those references mechanically, from `step_outputs` — the raw-result store notebook 2 set up, and the reason a float can still be recovered after the observation became a sentence

✓ Separated `plan_progress` (decided) from `plan_done` (executed), avoiding a loop that exits with its final step decided and never run

✓ Ran the plan with `Loop(until=plan_done)` — the predicate form, for when the number of rounds isn't knowable in advance — and watched `DebugThink` repair two genuinely invalid planner arguments without spending a line of context

✓ Added replanning through `ContinueOrFinish`, so committing to a plan up front stays a safe bet

✓ Read the dependency graph directly out of the plan's own `$stepN` references, and ran the same plan through `Graph` — each step starting when its own dependencies finished, with the observations arriving in completion order and a measured wall-clock win

**Next:** `05_sub_agents.ipynb` — splitting the agent rather than the plan: delegating to sub-agents that run their own loops, having them debate each other, and choosing between what they bring back.